In [ ]:
# !pip install git+https://github.com/lucacorbucci/flower
# !pip install seaborn
# !pip install dill
# !pip install ray

In [ ]:
import torch
import random
import numpy as np
from Utils.tabular_data_loader import prepare_tabular_data
import os
from torch import nn
from client import FlowerClient
import flwr as fl
import dill
from fed_avg import FedAvg
import logging
from logging import DEBUG, INFO
from client_manager import SimpleClientManager
from server import Server
from client import FlowerClient
from Utils.train_parameters import TrainParameters

In [ ]:
seed = 41
pool_size = 150
num_client_cpus = 1  # percentage of cpu assigned to each client
num_client_gpus = 0.2  # percentage of gpus assigned to each client. With 0.5, on each GPU we will run two clients
# these parameters are used to configure Ray and they are dependent on
# the machine we want to use to run the experiments
ray_num_cpus = 20
ray_num_gpus = 2
ram_memory = 16_000 * 1024 * 1024 * 2
# (optional) specify Ray config
ray_init_args = {
    "include_dashboard": False,
    "num_cpus": ray_num_cpus,
    "num_gpus": ray_num_gpus,
    "_memory": ram_memory,
    "_redis_max_memory": 100000000,
    "object_store_memory": 100000000,
    "logging_level": logging.ERROR,
    "log_to_driver": True,
}

In [ ]:
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)
torch.cuda.manual_seed_all(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

os.environ["PYTHONHASHSEED"] = str(seed)
current_max_epsilon = 0
pool_size = pool_size
client_resources = {
    "num_cpus": num_client_cpus,
    "num_gpus": num_client_gpus,
}

In [ ]:
# we prepare the dataset that will be used for the training

dataset_path = "../dataset/"
dataset_name = "dutch"
approach = "representative"
ratio_unfair_nodes = 0.5
opposite_direction = False
ratio_unfairness = (0.6, 0.8)
group_to_reduce = (0, 1)
group_to_increment = (1, 1)
number_of_samples_per_node = 343
fed_dir, _ = prepare_tabular_data(
    dataset_path=dataset_path,
    dataset_name=dataset_name,
    do_iid_split=True,
    approach=approach,
    num_nodes=pool_size,
    ratio_unfair_nodes=ratio_unfair_nodes,
    opposite_direction=opposite_direction,
    ratio_unfairness=ratio_unfairness,
    group_to_reduce=group_to_reduce,
    group_to_increment=group_to_increment,
    number_of_samples_per_node=number_of_samples_per_node,
    opposite_group_to_reduce=None,
    opposite_group_to_increment=None,
    opposite_ratio_unfairness=None,
    one_group_nodes=True,
)

In [ ]:
class LinearClassificationNet(nn.Module):
    """
    A fully-connected single-layer linear NN for classification.
    """

    def __init__(self, input_size=11, output_size=2):
        super(LinearClassificationNet, self).__init__()
        self.layer1 = nn.Linear(input_size, output_size, bias=False)

    def forward(self, x):
        x = self.layer1(x.float())
        return x

In [ ]:
# Functions that will be used during the training
def handle_counters(metrics, key):
    combinations = ["1|0", "1|1"]
    all_combinations = ["0|0", "0|1", "1|0", "1|1"]
    missing_combinations = [("0|0", "1|0"), ("0|1", "1|1")]
    targets = ["0", "1"]
    sum_counters = {"0|0": 0, "0|1": 0, "1|0": 0, "1|1": 0}
    sum_targets = {"0": 0, "1": 0}

    for _, metric in metrics:
        metric = metric[key]
        for combination in combinations:
            try:
                sum_counters[combination] += metric[combination]
            except:
                continue

        for target in targets:
            try:
                sum_targets[target] += metric[target]
            except:
                continue

    for non_existing, existing in missing_combinations:
        sum_counters[non_existing] = (
            sum_targets[existing[-1]] - sum_counters[existing]
            if sum_targets[existing[-1]] - sum_counters[existing] > 0
            else 0
        )
    average_probabilities = {}
    for combination in all_combinations:
        try:
            proba = sum_counters[combination] / sum_targets[combination[2]]
            if proba > 1:
                proba = 1
            if proba < 0:
                proba = 0
            average_probabilities[combination] = proba
        except:
            continue
    max_disparity_statistics = max(
        [
            sum_counters["0|0"] / sum_targets["0"] - sum_counters["0|1"] / sum_targets["1"],
            sum_counters["0|1"] / sum_targets["1"] - sum_counters["0|0"] / sum_targets["0"],
            sum_counters["1|0"] / sum_targets["0"] - sum_counters["1|1"] / sum_targets["1"],
            sum_counters["1|1"] / sum_targets["1"] - sum_counters["1|0"] / sum_targets["0"],
        ]
    )
    return (
        sum_counters,
        sum_targets,
        average_probabilities,
        max_disparity_statistics,
    )


def agg_metrics_test(metrics: list, server_round: int) -> dict:
    total_examples = sum([n_examples for n_examples, _ in metrics])

    loss_test = (
        sum(
            [
                n_examples * metric["test_loss" if not train_parameters.sweep else "validation_loss"]
                for n_examples, metric in metrics
            ]
        )
        / total_examples
    )
    accuracy_test = (
        sum(
            [
                n_examples * metric["test_accuracy" if not train_parameters.sweep else "validation_accuracy"]
                for n_examples, metric in metrics
            ]
        )
        / total_examples
    )
    f1_test = sum([n_examples * metric["f1_score"] for n_examples, metric in metrics]) / total_examples
    max_disparity_average = np.mean(
        [
            metric["max_disparity_test" if not train_parameters.sweep else "max_disparity_validation"]
            for n_examples, metric in metrics
        ]
    )
    # weighted average of the disparity of the different nodes
    max_disparity_weighted_average = (
        sum(
            [
                n_examples * metric["max_disparity_test" if not train_parameters.sweep else "max_disparity_validation"]
                for n_examples, metric in metrics
            ]
        )
        / total_examples
    )

    # Log data from the different test clients:
    for _, metric in metrics:
        node_name = metric["cid"]
        disparity = metric["max_disparity_test" if not train_parameters.sweep else "max_disparity_validation"]
        accuracy = metric["test_accuracy" if not train_parameters.sweep else "validation_accuracy"]
        disparity_dataset = metric["max_disparity_dataset"]
        agg_metrics = {
            f"Test Node {node_name} - Acc.": accuracy,
            f"Test Node {node_name} - Disp.": disparity,
            f"Test Node {node_name} - Disp. Dataset": disparity_dataset,
            "FL Round": server_round,
        }

    (
        sum_counters,
        sum_targets,
        average_probabilities,
        max_disparity_statistics,
    ) = handle_counters(metrics, "counters")

    # write avg probabilities to file
    with open(f"{fed_dir}/avg_proba.pkl", "wb") as file:
        dill.dump(average_probabilities, file)

    print(
        f"========> Metrics on test set: Accuracy {accuracy_test} - Disparity with statistics {max_disparity_statistics}"
    )

    agg_metrics = {
        "Test Loss": loss_test,
        "Test Accuracy": accuracy_test,
        "Test Disparity with average": max_disparity_average,
        "Test Disparity with weighted average": max_disparity_weighted_average,
        "Test Disparity with statistics": max_disparity_statistics,
        "FL Round": server_round,
        "Test Counter 0|0": sum_counters["0|0"],
        "Test Counter 0|1": sum_counters["0|1"],
        "Test Counter 1|0": sum_counters["1|0"],
        "Test Counter 1|1": sum_counters["1|1"],
        "Test Target 0": sum_targets["0"],
        "Test Target 1": sum_targets["1"],
        "Test F1": f1_test,
    }

    return agg_metrics


def agg_metrics_train(metrics: list, server_round: int, current_max_epsilon: float, fed_dir) -> dict:
    # Collect the losses logged during each epoch in each client
    total_examples = sum([n_examples for n_examples, _ in metrics])

    losses = []
    losses_with_regularization = []
    epsilon_list = []
    accuracies = []
    lambda_list = []
    max_disparity_train = []

    for n_examples, node_metrics in metrics:
        losses.append(n_examples * node_metrics["train_loss"])

        losses_with_regularization.append(n_examples * node_metrics["train_loss_with_regularization"])
        epsilon_list.append(node_metrics["epsilon"])
        accuracies.append(n_examples * node_metrics["train_accuracy"])
        lambda_list.append(node_metrics["Lambda"])
        disparity = node_metrics["Max Disparity Dataset"]
        client_id = node_metrics["cid"]
        disparity_client_after_local_epoch = node_metrics["Disparity Train"]
        max_disparity_train.append(disparity_client_after_local_epoch)
        disparity_client_before_local_epoch = node_metrics["Max Disparity Train Before Local Epoch"]

        DPL_lambda = node_metrics["Lambda"]

        # Create the dictionary we want to log. For some metrics we want to log
        # we have to check if they are present or not.
        to_be_logged = {
            f"Disparity Client {client_id} After Local train": disparity_client_after_local_epoch,
            f"Disparity Client {client_id} Before local train": disparity_client_before_local_epoch,
            "FL Round": server_round,
        }
        if disparity:
            to_be_logged[f"Disparity Dataset Client {client_id}"] = disparity
        if DPL_lambda:
            to_be_logged[f"Lambda Client {client_id}"] = DPL_lambda
        delta = node_metrics["delta"]
        if delta:
            to_be_logged[f"Delta Client {client_id}"] = delta

    # weighted average of the disparity of the different nodes
    max_disparity_weighted_average = (
        sum([n_examples * metric["Disparity Train"] for n_examples, metric in metrics]) / total_examples
    )

    (
        sum_counters,
        sum_targets,
        average_probabilities,
        max_disparity_statistics,
    ) = handle_counters(metrics, "counters")

    (
        sum_counters_no_noise,
        sum_targets_no_noise,
        _,
        max_disparity_statistics_no_noise,
    ) = handle_counters(metrics, "counters_no_noise")

    current_max_epsilon = max(current_max_epsilon, *epsilon_list)
    agg_metrics = {
        "Train Loss": sum(losses) / total_examples,
        "Train Accuracy": sum(accuracies) / total_examples,
        "Train Loss with Regularization": sum(losses_with_regularization) / total_examples,
        "Average Probabilities": average_probabilities,
        "Training Disparity with average": sum(max_disparity_train) / len(max_disparity_train),
        "Training Disparity with weighted average": max_disparity_weighted_average,
        "Aggregated Lambda": sum(lambda_list) / len(lambda_list),
        "Train Epsilon": current_max_epsilon,
        "FL Round": server_round,
    }

    print(f"Aggregated Lambda: {sum(lambda_list) / len(lambda_list)}")
    print("Max Disparity Statistics Train: ", max_disparity_statistics)
    print("Max Disparity Avg Train: ", sum(max_disparity_train) / len(max_disparity_train))

    return agg_metrics

# Training a model with unfairness mitigation with Fixed Lambda + Differential Privacy

In [ ]:
# DPL_value = 0.24589232391680177
starting_lambda_mode = "fixed"  # we use the fixed lambda version of DPL here+
starting_lambda_value = 0.24589232391680177
num_training_nodes = 100
num_test_nodes = 50
batch_size = 214
fl_rounds = 39
epochs = 2
lr = 0.04554998744914608
clipping = 100000000  # big value because now we do not want to use DP
sampled_clients = 0.3
sampled_clients_test = 1.0
target = 0.1

In [ ]:
# We create the train parameters object that will be passed to the clients
train_parameters = TrainParameters(
    epochs=epochs,
    device="cuda" if torch.cuda.is_available() else "cpu",
    batch_size=batch_size,
    seed=seed,
    optimizer="adam",
    regularization=True,
    regularization_lambda=starting_lambda_value,
    regularization_mode="fixed",
    target=target,
    # epsilon=0.75,
    # epsilon_statistics=0.25,
    fl_round=fl_rounds,
)

In [ ]:
# Function that is called to create the clients
def client_fn(cid: str):
    # create a single client instance
    return FlowerClient(
        train_parameters=train_parameters,
        cid=cid,
        fed_dir_data=fed_dir,
        dataset_name=dataset_name,
        clipping=clipping,
        # delta=args.delta,
        lr=lr,
    )

In [ ]:
model = LinearClassificationNet(input_size=11, output_size=2)
model_parameters = [val.cpu().numpy() for _, val in model.state_dict().items()]
initial_parameters = fl.common.ndarrays_to_parameters(model_parameters)

In [ ]:
def fit_config(server_round: int = 0):
    """Return a configuration with static batch size and (local) epochs."""
    config = {
        "epochs": epochs,  # number of local epochs
        "batch_size": batch_size,
        "dataset": dataset_name,
        "server_round": server_round,
    }
    return config


def evaluate_config(server_round: int = 0):
    """Return a configuration with static batch size and (local) epochs."""
    config = {
        "epochs": epochs,  # number of local epochs
        "batch_size": batch_size,
        "dataset": dataset_name,
    }
    return config

In [ ]:
# define the strategy that will be used to train the federated model
strategy = FedAvg(
    fraction_fit=sampled_clients,
    fraction_evaluate=0,
    fraction_test=sampled_clients_test,
    min_fit_clients=sampled_clients,
    min_evaluate_clients=0,
    min_available_clients=sampled_clients,
    on_fit_config_fn=fit_config,
    on_evaluate_config_fn=evaluate_config,
    initial_parameters=initial_parameters,
    fit_metrics_aggregation_fn=agg_metrics_train,
    evaluate_metrics_aggregation_fn=None,
    test_metrics_aggregation_fn=agg_metrics_test,
    current_max_epsilon=current_max_epsilon,
    fed_dir=fed_dir,
)

In [ ]:
client_manager = SimpleClientManager(
    seed=seed,
    num_clients=pool_size,
    sort_clients=True,
    num_training_nodes=num_training_nodes,
    num_validation_nodes=0,
    num_test_nodes=num_test_nodes,
    node_shuffle_seed=243798357,
    fed_dir=fed_dir,
    ratio_unfair_nodes=ratio_unfair_nodes,
    fl_rounds=fl_rounds,
    fraction_fit=sampled_clients,
    fraction_evaluate=0,
    fraction_test=sampled_clients_test,
)

In [ ]:
server = Server(client_manager=client_manager, strategy=strategy)

In [ ]:
fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=pool_size,
    client_resources=client_resources,
    config=fl.server.ServerConfig(num_rounds=fl_rounds),
    strategy=strategy,
    ray_init_args=ray_init_args,
    server=server,
    client_manager=client_manager,
)

# Training without Unfairness Removal

In [ ]:
DPL_value = 0
starting_lambda_mode = "fixed"  # we use the fixed lambda version of DPL here
num_training_nodes = 100
num_test_nodes = 50
batch_size = 278
fl_rounds = 3
epochs = 39
lr = 0.0460886474538264
clipping = 100000000  # big value because now we do not want to use DP
sampled_clients = 0.3
sampled_clients_test = 0.5

In [ ]:
# We create the train parameters object that will be passed to the clients
train_parameters = TrainParameters(
    epochs=epochs,
    device="cuda" if torch.cuda.is_available() else "cpu",
    batch_size=batch_size,
    seed=seed,
    optimizer="adam",
    regularization=False,
    fl_round=fl_rounds,
)

In [ ]:
# Function that is called to create the clients
def client_fn(cid: str):
    # create a single client instance
    return FlowerClient(
        train_parameters=train_parameters,
        cid=cid,
        fed_dir_data=fed_dir,
        dataset_name=dataset_name,
        clipping=clipping,
        # delta=args.delta,
        lr=lr,
    )

In [ ]:
model = LinearClassificationNet(input_size=11, output_size=2)
model_parameters = [val.cpu().numpy() for _, val in model.state_dict().items()]
initial_parameters = fl.common.ndarrays_to_parameters(model_parameters)

In [ ]:
def fit_config(server_round: int = 0):
    """Return a configuration with static batch size and (local) epochs."""
    config = {
        "epochs": epochs,  # number of local epochs
        "batch_size": batch_size,
        "dataset": dataset_name,
        "server_round": server_round,
    }
    return config


def evaluate_config(server_round: int = 0):
    """Return a configuration with static batch size and (local) epochs."""
    config = {
        "epochs": epochs,  # number of local epochs
        "batch_size": batch_size,
        "dataset": dataset_name,
    }
    return config

In [ ]:
# define the strategy that will be used to train the federated model
strategy = FedAvg(
    fraction_fit=sampled_clients,
    fraction_evaluate=0,
    fraction_test=sampled_clients_test,
    min_fit_clients=sampled_clients,
    min_evaluate_clients=0,
    min_available_clients=sampled_clients,
    on_fit_config_fn=fit_config,
    on_evaluate_config_fn=evaluate_config,
    initial_parameters=initial_parameters,
    fit_metrics_aggregation_fn=agg_metrics_train,
    evaluate_metrics_aggregation_fn=None,
    test_metrics_aggregation_fn=agg_metrics_test,
    current_max_epsilon=current_max_epsilon,
    fed_dir=fed_dir,
)

In [ ]:
client_manager = SimpleClientManager(
    seed=seed,
    num_clients=pool_size,
    sort_clients=True,
    num_training_nodes=num_training_nodes,
    num_validation_nodes=0,
    num_test_nodes=num_test_nodes,
    node_shuffle_seed=123,
    fed_dir=fed_dir,
    ratio_unfair_nodes=ratio_unfair_nodes,
    fl_rounds=fl_rounds,
    fraction_fit=sampled_clients,
    fraction_evaluate=0,
    fraction_test=sampled_clients_test,
)

In [ ]:
server = Server(client_manager=client_manager, strategy=strategy)

In [ ]:
fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=pool_size,
    client_resources=client_resources,
    config=fl.server.ServerConfig(num_rounds=fl_rounds),
    strategy=strategy,
    ray_init_args=ray_init_args,
    server=server,
    client_manager=client_manager,
)

# Training a model with unfairness mitigation and Tunable Lambda

In [ ]:
DPL_value = 0.24589232391680177
starting_lambda_value = 0.24589232391680177
num_training_nodes = 100
num_test_nodes = 50
batch_size = 214
fl_rounds = 5
epochs = 3
lr = 0.04554998744914608
clipping = 100000000  # big value because now we do not want to use DP
sampled_clients = 0.5
sampled_clients_test = 1.0
target = 0.1
momentum = 0.50690219349591
alpha = 3.645210016895725

In [ ]:
# We create the train parameters object that will be passed to the clients
train_parameters = TrainParameters(
    epochs=epochs,
    device="cuda" if torch.cuda.is_available() else "cpu",
    batch_size=batch_size,
    seed=seed,
    optimizer="adam",
    regularization=True,
    regularization_mode="tunable",
    momentum=momentum,
    alpha=alpha,
    weight_decay_alpha=0.9324356306287443,
    target=target,
    fl_round=fl_rounds,
)

In [ ]:
# Function that is called to create the clients
def client_fn(cid: str):
    # create a single client instance
    return FlowerClient(
        train_parameters=train_parameters,
        cid=cid,
        fed_dir_data=fed_dir,
        dataset_name=dataset_name,
        clipping=clipping,
        # delta=args.delta,
        lr=lr,
    )

In [ ]:
model = LinearClassificationNet(input_size=11, output_size=2)
model_parameters = [val.cpu().numpy() for _, val in model.state_dict().items()]
initial_parameters = fl.common.ndarrays_to_parameters(model_parameters)

In [ ]:
def fit_config(server_round: int = 0):
    """Return a configuration with static batch size and (local) epochs."""
    config = {
        "epochs": epochs,  # number of local epochs
        "batch_size": batch_size,
        "dataset": dataset_name,
        "server_round": server_round,
    }
    return config


def evaluate_config(server_round: int = 0):
    """Return a configuration with static batch size and (local) epochs."""
    config = {
        "epochs": epochs,  # number of local epochs
        "batch_size": batch_size,
        "dataset": dataset_name,
    }
    return config

In [ ]:
# define the strategy that will be used to train the federated model
strategy = FedAvg(
    fraction_fit=sampled_clients,
    fraction_evaluate=0,
    fraction_test=sampled_clients_test,
    min_fit_clients=sampled_clients,
    min_evaluate_clients=0,
    min_available_clients=sampled_clients,
    on_fit_config_fn=fit_config,
    on_evaluate_config_fn=evaluate_config,
    initial_parameters=initial_parameters,
    fit_metrics_aggregation_fn=agg_metrics_train,
    evaluate_metrics_aggregation_fn=None,
    test_metrics_aggregation_fn=agg_metrics_test,
    current_max_epsilon=current_max_epsilon,
    fed_dir=fed_dir,
)

In [ ]:
client_manager = SimpleClientManager(
    seed=seed,
    num_clients=pool_size,
    sort_clients=True,
    num_training_nodes=num_training_nodes,
    num_validation_nodes=0,
    num_test_nodes=num_test_nodes,
    node_shuffle_seed=243798356,
    fed_dir=fed_dir,
    ratio_unfair_nodes=ratio_unfair_nodes,
    fl_rounds=fl_rounds,
    fraction_fit=sampled_clients,
    fraction_evaluate=0,
    fraction_test=sampled_clients_test,
)

In [ ]:
server = Server(client_manager=client_manager, strategy=strategy)

In [ ]:
fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=pool_size,
    client_resources=client_resources,
    config=fl.server.ServerConfig(num_rounds=fl_rounds),
    strategy=strategy,
    ray_init_args=ray_init_args,
    server=server,
    client_manager=client_manager,
)

# Training a model with unfairness mitigation, Differential Privacy and Tunable Lambda

In [ ]:
DPL_value = 0.24589232391680177
starting_lambda_value = 0.24589232391680177
num_training_nodes = 100
num_test_nodes = 50
batch_size = 214
fl_rounds = 5
epochs = 39
lr = 0.04554998744914608
clipping = 100000000  # big value because now we do not want to use DP
sampled_clients = 0.5
sampled_clients_test = 1.0
target = 0.1
momentum = 0.50690219349591
alpha = 3.645210016895725

In [ ]:
# We create the train parameters object that will be passed to the clients
train_parameters = TrainParameters(
    epochs=epochs,
    device="cuda" if torch.cuda.is_available() else "cpu",
    batch_size=batch_size,
    seed=seed,
    optimizer="adam",
    regularization=True,
    regularization_mode="tunable",
    momentum=momentum,
    alpha=alpha,
    weight_decay_alpha=0.9324356306287443,
    target=target,
    fl_round=fl_rounds,
    epsilon=2.0,
    epsilon_statistics=0.5,
    epsilon_lambda=0.5,
)

In [ ]:
# Function that is called to create the clients
def client_fn(cid: str):
    # create a single client instance
    return FlowerClient(
        train_parameters=train_parameters,
        cid=cid,
        fed_dir_data=fed_dir,
        dataset_name=dataset_name,
        clipping=clipping,
        # delta=args.delta,
        lr=lr,
    )

In [ ]:
model = LinearClassificationNet(input_size=11, output_size=2)
model_parameters = [val.cpu().numpy() for _, val in model.state_dict().items()]
initial_parameters = fl.common.ndarrays_to_parameters(model_parameters)

In [ ]:
def fit_config(server_round: int = 0):
    """Return a configuration with static batch size and (local) epochs."""
    config = {
        "epochs": epochs,  # number of local epochs
        "batch_size": batch_size,
        "dataset": dataset_name,
        "server_round": server_round,
    }
    return config


def evaluate_config(server_round: int = 0):
    """Return a configuration with static batch size and (local) epochs."""
    config = {
        "epochs": epochs,  # number of local epochs
        "batch_size": batch_size,
        "dataset": dataset_name,
    }
    return config

In [ ]:
# define the strategy that will be used to train the federated model
strategy = FedAvg(
    fraction_fit=sampled_clients,
    fraction_evaluate=0,
    fraction_test=sampled_clients_test,
    min_fit_clients=sampled_clients,
    min_evaluate_clients=0,
    min_available_clients=sampled_clients,
    on_fit_config_fn=fit_config,
    on_evaluate_config_fn=evaluate_config,
    initial_parameters=initial_parameters,
    fit_metrics_aggregation_fn=agg_metrics_train,
    evaluate_metrics_aggregation_fn=None,
    test_metrics_aggregation_fn=agg_metrics_test,
    current_max_epsilon=current_max_epsilon,
    fed_dir=fed_dir,
)

In [ ]:
client_manager = SimpleClientManager(
    seed=seed,
    num_clients=pool_size,
    sort_clients=True,
    num_training_nodes=num_training_nodes,
    num_validation_nodes=0,
    num_test_nodes=num_test_nodes,
    node_shuffle_seed=243798356,
    fed_dir=fed_dir,
    ratio_unfair_nodes=ratio_unfair_nodes,
    fl_rounds=fl_rounds,
    fraction_fit=sampled_clients,
    fraction_evaluate=0,
    fraction_test=sampled_clients_test,
)

In [ ]:
server = Server(client_manager=client_manager, strategy=strategy)

In [ ]:
fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=pool_size,
    client_resources=client_resources,
    config=fl.server.ServerConfig(num_rounds=fl_rounds),
    strategy=strategy,
    ray_init_args=ray_init_args,
    server=server,
    client_manager=client_manager,
)

# Comparison

| Unfairness Mitigation  | Accuracy  | Final Disparity  |
|---|---|---|
| False  | 0.8050 | 0.2365 |
| True | 0.6328  | 0.1191 |

Please note that the results may be improved with an hyperparameter search. For these two examples I've used the hyperapameters of a different experiment.